# Demo: what SFT can and cannot do for clinical transcript extraction

This notebook walks through four checkpoints (v1, v2, v2.1-lite, v3-DPO) on four templates (SOAP, referral_a, referral_b, MSE). For each checkpoint we show what it fixes from the previous one and what new problem it exposes. The progression ends with two clearly separated problems that remain after SFT:

1. **Problem 1 (coverage).** Templates the model has never been trained on. The fix is more training data.
2. **Problem 2 (behavioural).** On templates the model has been trained on, the JSON is well-formed but the content has misses or invented spans. SFT's token loss cannot see this. The fix is preference learning (DPO, then GRPO).

**This notebook closes the loop.** v3 is the DPO checkpoint, v2.1-lite further trained on verifier-scored preference pairs. We measure v3 against v2.1 on the in-distribution templates to see whether preference learning moved the Problem 2 metrics, and we report the result as-is. DPO targets Problem 2 only. It is not expected to move Problem 1.

## Definitions in one place

**Templates** are clinical output schemas. We use four: `soap` (SOAP note), `referral_a` (referral letter style A), `referral_b` (referral letter style B), `mse` (Mental State Examination).

**Coverage** of a checkpoint is the set of templates it saw during supervised fine-tuning (SFT). Anything outside that set is out-of-distribution (OOD) for the checkpoint.

**Problem 1 (coverage).** The model has never been trained on a given template. No method change fixes this. The fix is more training data on that template.

**Problem 2 (behavioural).** The model has been trained on the template, the output is well-formed JSON, but the content has misses or hallucinations (invented ungrounded spans). Token-level cross-entropy in SFT cannot see this, because both the wrong output and the right output are valid sequences of tokens. The fix is whole-output preference learning (DPO, then GRPO).

**Evaluation terms used throughout.**

| term | what it means |
|---|---|
| `schema_valid` | output parses as JSON and matches the template schema |
| `content_overlap` | Jaccard overlap between predicted populated fields and gold populated fields |
| `wrong_null_rate` | fraction of gold-populated fields where the model emitted null (a miss) |
| `correct_null_rate` | fraction of gold-null fields where the model correctly emitted null |
| `ungrounded_span_rate` | fraction of predicted spans not found in the transcript (hallucinations) |
| `all_null_rate` | fraction of outputs that are entirely null (degenerate) |


### How to read this notebook

Setup -> Verifier -> Run v1 -> Obs v1 -> Run v2 -> Obs v2 -> Run v2.1 -> Obs v2.1 -> Run v3 (DPO) -> Obs v3 -> Combined progression -> Problem 1 (table + obs + MSE qualitative) -> Problem 2 (v2.1 in-dist stats + 3 examples + DPO motivation) -> Problem 2 revisited (v2.1 vs v3 + side-by-side diffs) -> Recap -> What comes next.

## Setup

Standard environment probe, then load paths to the three GGUF models and the four eval data files. No model is loaded yet.

In [1]:
import importlib.metadata as metadata
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path


def run_text(command):
    try:
        result = subprocess.run(command, capture_output=True, text=True, check=False)
        text = (result.stdout or result.stderr).strip()
        return text or "<no output>"
    except Exception as exc:
        return f"<unavailable: {exc}>"


print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Machine:", platform.machine())
print("CPU cores:", os.cpu_count())

if sys.platform == "darwin":
    mem_bytes = run_text(["sysctl", "-n", "hw.memsize"])
    if mem_bytes.isdigit():
        print("Unified memory (GB):", round(int(mem_bytes) / (1024 ** 3), 2))
elif shutil.which("nvidia-smi"):
    print("\nGPU info:")
    print(run_text(["nvidia-smi"]))

print("llama-cpp-python:", metadata.version("llama-cpp-python"))

from llama_cpp import Llama, llama_cpp as lib

GPU_OFFLOAD_SUPPORTED = bool(lib.llama_supports_gpu_offload())
AUTO_N_GPU_LAYERS = -1 if GPU_OFFLOAD_SUPPORTED else 0
print("GPU offload supported:", GPU_OFFLOAD_SUPPORTED)
print("Default n_gpu_layers:", AUTO_N_GPU_LAYERS)

Python: 3.12.12
Platform: macOS-26.5-arm64-arm-64bit
Machine: arm64
CPU cores: 10
Unified memory (GB): 16.0
llama-cpp-python: 0.3.21


ggml_metal_device_init: tensor API disabled for pre-M5 and pre-A19 devices
ggml_metal_library_init: using embedded metal library
ggml_metal_library_init: loaded in 0.031 sec
ggml_metal_rsets_init: creating a residency set collection (keep_alive = 180 s)
ggml_metal_device_init: GPU name:   MTL0 (Apple M2 Pro)
ggml_metal_device_init: GPU family: MTLGPUFamilyApple8  (1008)
ggml_metal_device_init: GPU family: MTLGPUFamilyCommon3 (3003)
ggml_metal_device_init: GPU family: MTLGPUFamilyMetal4  (5002)
ggml_metal_device_init: simdgroup reduction   = true
ggml_metal_device_init: simdgroup matrix mul. = true
ggml_metal_device_init: has unified memory    = true
ggml_metal_device_init: has bfloat            = true
ggml_metal_device_init: has tensor            = false
ggml_metal_device_init: use residency sets    = true
ggml_metal_device_init: use shared buffers    = true
ggml_metal_device_init: recommendedMaxWorkingSetSize  = 12713.12 MB


GPU offload supported: True
Default n_gpu_layers: -1


In [2]:
import html
import json
import re
import time
from functools import lru_cache

from IPython.display import HTML, display

from src.data_generation.templates import REGISTRY
from src.data_generation.validate import (
    _check_mse,
    _check_referral,
    _check_soap,
    _collect_spans,
)
from src.prompts import build_inference_messages


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "models").exists():
            return candidate
    raise FileNotFoundError("Could not locate repo root from current working directory")


REPO_ROOT = find_repo_root(Path.cwd())

MODELS = {
    "v1":   REPO_ROOT / "models" / "v1_qwen3b-soap-q4_k_m.gguf",
    "v2":   REPO_ROOT / "models" / "v2_qwen3b-multitemplate-q4_k_m.gguf",
    "v2.1": REPO_ROOT / "models" / "v2_1_lite-qwen3b-multitemplate-q4_k_m.gguf",
    "v3":   REPO_ROOT / "models" / "v3_qwen3b-multitemplate_dpo-q4_k_m.gguf",
}
DATA_FILES = {
    "soap":          REPO_ROOT / "data" / "qwen3.5_latest" / "eval_in_dist.soap.jsonl",
    "referral_a":    REPO_ROOT / "data" / "qwen3.5_latest" / "eval_in_dist.referral_a.jsonl",
    "referral_b":    REPO_ROOT / "data" / "qwen3.5_latest" / "eval_zero_shot.referral_b.jsonl",
    "mse_zero_shot": REPO_ROOT / "data" / "qwen3.5_latest" / "eval_zero_shot.mse.jsonl",
    "mse_in_dist":   REPO_ROOT / "data" / "qwen3.5_latest" / "eval_in_dist.mse.jsonl",
}

# Per-model evaluation plan.
# soap / referral_a / referral_b are directly comparable across all three models.
# mse is a staged comparison: v1 and v2 are zero-shot on mse; v2.1 is in-distribution on mse.
EVAL_PLAN = {
    "v1": [
        {"template": "soap",       "split": "in_dist",   "file_key": "soap",          "note": "v1 was trained on SOAP. Expect strongest performance here."},
        {"template": "referral_a", "split": "in_dist",   "file_key": "referral_a",    "note": "Never trained on referral_a. Tests prompt-time schema injection alone."},
        {"template": "referral_b", "split": "held_out",  "file_key": "referral_b",    "note": "Never trained on referral_b. Style-near unseen template."},
        {"template": "mse",        "split": "zero_shot", "file_key": "mse_zero_shot", "note": "Never trained on MSE. Genuinely unseen ontology."},
    ],
    "v2": [
        {"template": "soap",       "split": "in_dist",   "file_key": "soap",          "note": "In-distribution. Should hold up after multi-template training."},
        {"template": "referral_a", "split": "in_dist",   "file_key": "referral_a",    "note": "In-distribution. The second template v2 was trained on."},
        {"template": "referral_b", "split": "held_out",  "file_key": "referral_b",    "note": "Unseen referral variant. Does referral_a training transfer?"},
        {"template": "mse",        "split": "zero_shot", "file_key": "mse_zero_shot", "note": "Still genuinely unseen ontology. This is far-OOD for v1 and v2. Used as Problem 1 evidence."},
    ],
    "v2.1": [
        {"template": "soap",       "split": "in_dist",   "file_key": "soap",          "note": "In-distribution. Check we did not regress SOAP after adding MSE."},
        {"template": "referral_a", "split": "in_dist",   "file_key": "referral_a",    "note": "In-distribution. Check referral_a still works after adding MSE."},
        {"template": "referral_b", "split": "held_out",  "file_key": "referral_b",    "note": "Still unseen. This is the honest Problem 1 probe for v2.1."},
        {"template": "mse",        "split": "in_dist",   "file_key": "mse_in_dist",   "note": "Now in-distribution because v2.1 was trained on MSE. Measures whether SFT learned the ontology when given data, not whether it generalises to it."},
    ],
    "v3": [
        {"template": "soap",       "split": "in_dist",   "file_key": "soap",          "note": "In-distribution. Check DPO did not regress SOAP schema validity or content."},
        {"template": "referral_a", "split": "in_dist",   "file_key": "referral_a",    "note": "In-distribution. A core target for the DPO miss reduction."},
        {"template": "referral_b", "split": "held_out",  "file_key": "referral_b",    "note": "Still unseen. Stretch hypothesis: DPO carry-over from referral_a. Reported as-is."},
        {"template": "mse",        "split": "in_dist",   "file_key": "mse_in_dist",   "note": "In-distribution. The lowest-overlap trained template, so the clearest DPO test."},
    ],
}

MODEL_ORDER = ["v1", "v2", "v2.1", "v3"]
DIRECT_TEMPLATES = ["soap", "referral_a", "referral_b"]
TEMPLATE_ORDER = ["soap", "referral_a", "referral_b", "mse"]

N_CTX = 4096
MAX_TOKENS = 512
N_THREADS = max(1, (os.cpu_count() or 4) - 2)
N_GPU_LAYERS = AUTO_N_GPU_LAYERS

for path in [*MODELS.values(), *DATA_FILES.values()]:
    assert path.exists(), f"Missing required file: {path}"

print("Repo root:", REPO_ROOT)
print("Models:")
for name, path in MODELS.items():
    print(f"  {name}: {path.name} ({path.stat().st_size / 1e9:.2f} GB)")
print("Data files:")
for name, path in DATA_FILES.items():
    print(f"  {name}: {path.relative_to(REPO_ROOT)}")

Repo root: /Users/nizamuddin/clinical_transcript_summariser
Models:
  v1: v1_qwen3b-soap-q4_k_m.gguf (1.93 GB)
  v2: v2_qwen3b-multitemplate-q4_k_m.gguf (1.93 GB)
  v2.1: v2_1_lite-qwen3b-multitemplate-q4_k_m.gguf (1.93 GB)
  v3: v3_qwen3b-multitemplate_dpo-q4_k_m.gguf (1.93 GB)
Data files:
  soap: data/qwen3.5_latest/eval_in_dist.soap.jsonl
  referral_a: data/qwen3.5_latest/eval_in_dist.referral_a.jsonl
  referral_b: data/qwen3.5_latest/eval_zero_shot.referral_b.jsonl
  mse_zero_shot: data/qwen3.5_latest/eval_zero_shot.mse.jsonl
  mse_in_dist: data/qwen3.5_latest/eval_in_dist.mse.jsonl


## Evaluation: *The verifier*

Every result in this notebook comes from the same verifier. We deliberately keep it in one place because the same function will later be used as the reward signal for post-training (DPO / GRPO). Eval and post-training will share one source of truth.

Metrics, in plain language:

- **`json_parse_rate`**: did the output parse as JSON at all. Higher is better.
- **`schema_valid_rate`**: did the JSON contain the required keys and nesting for the template. Higher is better.
- **`content_overlap_mean`**: how much of the gold content appears in the predicted content (token overlap on populated string fields). Higher is better. Useful within a template, not across templates.
- **`evidence_grounding_rate`**: of the `evidence_span` strings the model emitted, what fraction are exact substrings of the transcript. Higher is better. Low values mean the model invented evidence.
- **`ungrounded_span_rate`**: the inverse of grounding, surfaced explicitly because hallucination is what we are watching for. Lower is better.
- **`all_null_rate`**: fraction of outputs where every content field is empty or null. Higher is worse, but only if the transcripts actually had content (see next two).
- **`wrong_null_rate`**: of the gold fields that were populated, what fraction did the model leave null or empty (a miss). **This is the metric that operationalises degenerate behaviour on unseen schemas.** Higher is worse.
- **`correct_null_rate`**: of the gold fields that were empty, what fraction did the model also leave empty. Higher is better. This is the honest case where there was no evidence to extract.
- **`mean_latency_ms`**: average wall-clock per generation. Lower is better, but only matters once quality is acceptable.

`wrong_null_rate` and `correct_null_rate` together separate the honest case (no evidence, said nothing) from the miss case (evidence present, said nothing anyway).

In [3]:
_OBJECT_RE = re.compile(r"\{.*\}", re.DOTALL)
_FENCE_RE = re.compile(r"^```(?:json)?\s*\n?(.*?)\n?```$", re.DOTALL)

EXPECTED_TOP_KEYS = {
    "soap":       {"subjective", "objective", "assessment", "plan"},
    "referral_a": {"specialty", "patient", "reason", "history", "current_meds", "request"},
    "referral_b": {"specialty", "patient", "reason", "history", "current_meds", "request"},
    "mse":        {"appearance", "behaviour", "speech", "mood", "affect", "thought", "cognition", "insight"},
}
SCHEMA_CHECKERS = {
    "soap":       _check_soap,
    "referral_a": _check_referral,
    "referral_b": _check_referral,
    "mse":        _check_mse,
}


def unwrap_json_object(raw: str) -> str:
    raw = raw.strip()
    fence = _FENCE_RE.match(raw)
    if fence:
        raw = fence.group(1).strip()
    match = _OBJECT_RE.search(raw)
    return match.group(0) if match else raw


def parse_prediction(raw: str):
    try:
        return json.loads(unwrap_json_object(raw))
    except Exception:
        return None


# --- content-leaf traversal -------------------------------------------------
# Content leaves are the human-readable string values inside our schemas.
# We treat the three documented evidence-bearing shapes specially:
#   {"text":   ..., "evidence_span": ...}
#   {"name":   ..., "evidence_span": ...}
#   {"action": ..., "evidence_span": ...}
# Everything else is traversed recursively.

def walk_content_leaves(node, path=()):
    if isinstance(node, dict):
        if {"text", "evidence_span"}.issubset(node.keys()):
            yield path + ("text",), node.get("text")
            return
        if {"name", "evidence_span"}.issubset(node.keys()):
            yield path + ("name",), node.get("name")
            return
        if {"action", "evidence_span"}.issubset(node.keys()):
            yield path + ("action",), node.get("action")
            return
        for key, value in node.items():
            yield from walk_content_leaves(value, path + (key,))
    elif isinstance(node, list):
        for idx, item in enumerate(node):
            yield from walk_content_leaves(item, path + (idx,))


def is_filled(value) -> bool:
    return isinstance(value, str) and value.strip() != ""


def collect_content_values(node):
    return [value for _, value in walk_content_leaves(node)]


def content_token_set(node):
    tokens = set()
    for value in collect_content_values(node):
        if isinstance(value, str) and value.strip():
            tokens.update(value.lower().split())
    return tokens


def content_jaccard(pred_label, gold_label):
    pred_tokens = content_token_set(pred_label)
    gold_tokens = content_token_set(gold_label)
    if not pred_tokens and not gold_tokens:
        return 1.0
    return len(pred_tokens & gold_tokens) / max(len(pred_tokens | gold_tokens), 1)


def null_classification(pred_label, gold_label):
    """Compare pred vs gold at matching content-leaf paths.

    Returns counts: (gold_filled, gold_null, wrong_null, correct_null, over_populated).
      - wrong_null    : gold filled but pred null/missing  -> COLLAPSE
      - correct_null  : gold null     and pred null/missing -> correctly empty
      - over_populated: gold null but pred filled          -> potential hallucination
    """
    if not isinstance(pred_label, dict):
        pred_leaves = {}
    else:
        pred_leaves = {p: v for p, v in walk_content_leaves(pred_label)}

    gold_filled = gold_null = wrong_null = correct_null = over_populated = 0
    seen_paths = set()
    for path, gold_value in walk_content_leaves(gold_label):
        seen_paths.add(path)
        gold_is_filled = is_filled(gold_value)
        pred_is_filled = is_filled(pred_leaves.get(path))
        if gold_is_filled:
            gold_filled += 1
            if not pred_is_filled:
                wrong_null += 1
        else:
            gold_null += 1
            if pred_is_filled:
                over_populated += 1
            else:
                correct_null += 1

    # paths predicted but not in gold (extra structure) , count as over-population
    for path, pred_value in pred_leaves.items():
        if path in seen_paths:
            continue
        if is_filled(pred_value):
            over_populated += 1
    return {
        "gold_filled": gold_filled,
        "gold_null": gold_null,
        "wrong_null": wrong_null,
        "correct_null": correct_null,
        "over_populated": over_populated,
    }


def is_all_null_prediction(label):
    if not isinstance(label, dict):
        return True
    return all(not is_filled(value) for value in collect_content_values(label))


def grounding_counts(pred_label, transcript):
    if not isinstance(pred_label, dict):
        return 0, 0
    spans = [span for span in _collect_spans(pred_label) if span]
    grounded = sum(1 for span in spans if span in transcript)
    return grounded, len(spans)

In [4]:
@lru_cache(maxsize=None)
def load_llm(model_name: str):
    model_path = MODELS[model_name]
    print(f"[load] {model_name}: {model_path.name}")
    t0 = time.perf_counter()
    llm = Llama(
        model_path=str(model_path),
        n_ctx=N_CTX,
        n_gpu_layers=N_GPU_LAYERS,
        n_threads=N_THREADS,
        verbose=False,
        seed=0,
    )
    print(f"[load] ready in {time.perf_counter() - t0:.1f}s")
    return llm


def generate(model_name: str, template_name: str, transcript: str) -> str:
    llm = load_llm(model_name)
    messages = build_inference_messages(template_name, transcript)
    response = llm.create_chat_completion(
        messages=messages,
        temperature=0.0,
        max_tokens=MAX_TOKENS,
    )
    return response["choices"][0]["message"]["content"].strip()


def score_prediction(template_name: str, pred_raw: str, gold_label: dict, transcript: str):
    pred_label = parse_prediction(pred_raw)
    parse_ok = isinstance(pred_label, dict)
    schema_err = SCHEMA_CHECKERS[template_name](pred_label) if parse_ok else "unparseable"
    grounded, total_spans = grounding_counts(pred_label, transcript)
    null_stats = null_classification(pred_label, gold_label)
    return {
        "parse": int(parse_ok),
        "schema_valid": int(parse_ok and schema_err is None),
        "content_overlap": content_jaccard(pred_label, gold_label) if parse_ok else 0.0,
        "grounded_spans": grounded,
        "total_spans": total_spans,
        "all_null": int(is_all_null_prediction(pred_label)),
        "pred_label": pred_label,
        **null_stats,
    }


# --- tables ----------------------------------------------------------------

def print_table(rows, columns=None, digits=4):
    if not rows:
        print("No rows")
        return
    if columns is None:
        columns = list(rows[0].keys())

    def fmt(value):
        if isinstance(value, float):
            return f"{value:.{digits}f}"
        return str(value)

    widths = {col: max(len(col), max(len(fmt(row.get(col, ""))) for row in rows)) for col in columns}
    print(" | ".join(col.ljust(widths[col]) for col in columns))
    print("-+-".join("-" * widths[col] for col in columns))
    for row in rows:
        print(" | ".join(fmt(row.get(col, "")).ljust(widths[col]) for col in columns))


def summarise(model_name, template_name, split_name, details):
    n = len(details)
    gold_filled_total = sum(d["gold_filled"] for d in details)
    gold_null_total = sum(d["gold_null"] for d in details)
    total_spans = sum(d["total_spans"] for d in details)
    grounded_spans = sum(d["grounded_spans"] for d in details)
    grounding = (grounded_spans / total_spans) if total_spans else 1.0
    return {
        "model": model_name,
        "template": template_name,
        "split": split_name,
        "n": n,
        "json_parse_rate":         sum(d["parse"] for d in details) / n,
        "schema_valid_rate":       sum(d["schema_valid"] for d in details) / n,
        "content_overlap_mean":    sum(d["content_overlap"] for d in details) / n,
        "evidence_grounding_rate": grounding,
        "ungrounded_span_rate":    1.0 - grounding,
        "all_null_rate":           sum(d["all_null"] for d in details) / n,
        "wrong_null_rate":         (sum(d["wrong_null"]   for d in details) / gold_filled_total) if gold_filled_total else 0.0,
        "correct_null_rate":       (sum(d["correct_null"] for d in details) / gold_null_total)   if gold_null_total   else 1.0,
        "over_populated_total":    sum(d["over_populated"] for d in details),
        "mean_latency_ms":         sum(d["latency_ms"] for d in details) / n,
    }


DATASETS = {key: [json.loads(line) for line in path.open() if line.strip()] for key, path in DATA_FILES.items()}


def evaluate_model(model_name: str):
    summaries, all_details = [], []
    for item in EVAL_PLAN[model_name]:
        template_name = item["template"]
        split_name = item["split"]
        file_key = item["file_key"]
        rows = DATASETS[file_key]
        label_key = REGISTRY[template_name]["label_key"]
        template_details = []
        print(f"[run] {model_name} -> {template_name} ({split_name}, {len(rows)} rows) :: {item['note']}")
        for index, row in enumerate(rows, start=1):
            t0 = time.perf_counter()
            pred_raw = generate(model_name, template_name, row["transcript"])
            latency_ms = (time.perf_counter() - t0) * 1000
            scores = score_prediction(template_name, pred_raw, row[label_key], row["transcript"])
            detail = {
                "model": model_name,
                "template": template_name,
                "split": split_name,
                "file_key": file_key,
                "index": index,
                "transcript": row["transcript"],
                "gold": row[label_key],
                "pred_raw": pred_raw,
                "pred_label": scores.pop("pred_label"),
                "latency_ms": latency_ms,
                **scores,
            }
            template_details.append(detail)
            all_details.append(detail)
            print(
                f"  [{index:>2}/{len(rows)}] {latency_ms:>7.0f} ms  parse={detail['parse']} schema={detail['schema_valid']} "
                f"overlap={detail['content_overlap']:.2f} ground={(detail['grounded_spans']/detail['total_spans']) if detail['total_spans'] else 1.0:.2f} "
                f"all_null={detail['all_null']} wrong_null={detail['wrong_null']}/{detail['gold_filled']}"
            )
        summaries.append(summarise(model_name, template_name, split_name, template_details))
    return summaries, all_details


DISPLAY_COLUMNS = [
    "model", "template", "split", "n",
    "schema_valid_rate", "content_overlap_mean",
    "evidence_grounding_rate", "ungrounded_span_rate",
    "all_null_rate", "wrong_null_rate", "correct_null_rate",
    "mean_latency_ms",
]

## Run 1: model `v1` on all four templates

**Model**: v1, SFT on SOAP only. **Data**: 6 eval items per template (zero-shot for non-SOAP). **Why**: establish what a single-template SFT checkpoint does when faced with unseen schemas.

In [5]:
v1_summaries, v1_details = evaluate_model("v1")
print()
print_table(v1_summaries, columns=DISPLAY_COLUMNS)

[run] v1 -> soap (in_dist, 10 rows) :: v1 was trained on SOAP. Expect strongest performance here.
[load] v1: v1_qwen3b-soap-q4_k_m.gguf


llama_context: n_ctx_seq (4096) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


[load] ready in 1.4s
  [ 1/10]    6392 ms  parse=1 schema=1 overlap=0.31 ground=0.83 all_null=0 wrong_null=2/8
  [ 2/10]    2788 ms  parse=1 schema=1 overlap=0.32 ground=1.00 all_null=0 wrong_null=3/7
  [ 3/10]    3825 ms  parse=1 schema=1 overlap=0.24 ground=1.00 all_null=0 wrong_null=1/6
  [ 4/10]    7629 ms  parse=1 schema=1 overlap=0.33 ground=0.90 all_null=0 wrong_null=3/13
  [ 5/10]    5097 ms  parse=1 schema=1 overlap=0.44 ground=1.00 all_null=0 wrong_null=1/10
  [ 6/10]    4050 ms  parse=1 schema=1 overlap=0.41 ground=1.00 all_null=0 wrong_null=0/7
  [ 7/10]    4151 ms  parse=1 schema=1 overlap=0.30 ground=1.00 all_null=0 wrong_null=5/12
  [ 8/10]    3039 ms  parse=1 schema=1 overlap=0.18 ground=0.75 all_null=0 wrong_null=2/5
  [ 9/10]    5929 ms  parse=1 schema=1 overlap=0.38 ground=1.00 all_null=0 wrong_null=2/10
  [10/10]    3342 ms  parse=1 schema=1 overlap=0.50 ground=1.00 all_null=0 wrong_null=3/8
[run] v1 -> referral_a (in_dist, 10 rows) :: Never trained on referral_a. T

### Observation: v1

Schema validity is 1.00 across all four templates. Prompt-time schema injection works even on templates v1 never saw in SFT. That is the easy half.

Content is where v1 struggles, but not via degeneration. Content overlap is SOAP 0.34, referral_a 0.50, referral_b 0.27, MSE 0.05. On MSE, `wrong_null_rate` is 0.35 and `all_null_rate` is 0.00. v1 is guessing schema-shaped content rather than emitting nulls.

Takeaway: v1 is a weak generalist on unseen templates. It misses content. It does not yet show the degenerate-null failure we will see on v2.

## Run 2: model `v2` on all four templates

**Model**: v2, SFT on SOAP + referral_a. **Data**: same 6 items per template (zero-shot for MSE and referral_b). **Why**: see how adding a second template changes both trained and unseen-template behaviour.

In [6]:
v2_summaries, v2_details = evaluate_model("v2")
print()
print_table(v2_summaries, columns=DISPLAY_COLUMNS)

# delta vs v1, on the directly comparable templates
v1_lookup = {(r["template"], r["split"]): r for r in v1_summaries}
delta_rows = []
for r in v2_summaries:
    base = v1_lookup.get((r["template"], r["split"])) or v1_lookup.get((r["template"], "zero_shot"))
    if base is None:
        continue
    delta_rows.append({
        "template":          r["template"],
        "split":             r["split"],
        "d_schema_valid":    r["schema_valid_rate"]      - base["schema_valid_rate"],
        "d_overlap":         r["content_overlap_mean"]   - base["content_overlap_mean"],
        "d_grounding":       r["evidence_grounding_rate"]- base["evidence_grounding_rate"],
        "d_wrong_null":      r["wrong_null_rate"]        - base["wrong_null_rate"],
        "d_all_null":        r["all_null_rate"]          - base["all_null_rate"],
    })
print("\nDelta v2 - v1 (positive d_overlap/d_schema/d_grounding = improvement; positive d_wrong_null/d_all_null = regression):")
print_table(delta_rows)

[run] v2 -> soap (in_dist, 10 rows) :: In-distribution. Should hold up after multi-template training.
[load] v2: v2_qwen3b-multitemplate-q4_k_m.gguf


llama_context: n_ctx_seq (4096) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


[load] ready in 1.3s
  [ 1/10]    7789 ms  parse=1 schema=0 overlap=0.55 ground=0.90 all_null=0 wrong_null=3/8
  [ 2/10]    3017 ms  parse=1 schema=1 overlap=0.17 ground=1.00 all_null=0 wrong_null=3/7
  [ 3/10]    4139 ms  parse=1 schema=1 overlap=0.55 ground=1.00 all_null=0 wrong_null=0/6
  [ 4/10]    8238 ms  parse=1 schema=1 overlap=0.41 ground=0.91 all_null=0 wrong_null=2/13
  [ 5/10]    5317 ms  parse=1 schema=1 overlap=0.59 ground=1.00 all_null=0 wrong_null=1/10
  [ 6/10]    4769 ms  parse=1 schema=1 overlap=0.57 ground=1.00 all_null=0 wrong_null=1/7
  [ 7/10]    6529 ms  parse=1 schema=1 overlap=0.55 ground=1.00 all_null=0 wrong_null=1/12
  [ 8/10]    3641 ms  parse=1 schema=1 overlap=0.26 ground=0.83 all_null=0 wrong_null=1/5
  [ 9/10]    8132 ms  parse=1 schema=1 overlap=0.57 ground=0.89 all_null=0 wrong_null=1/10
  [10/10]    3901 ms  parse=1 schema=1 overlap=0.81 ground=1.00 all_null=0 wrong_null=1/8
[run] v2 -> referral_a (in_dist, 10 rows) :: In-distribution. The second te

### Observation: v2

v2 fixes v1's biggest gap (single-template coverage) and exposes the next one.

**Fixed.** Content overlap on trained templates: SOAP 0.34 -> 0.50, referral_a 0.50 -> 0.74. Schema validity on SOAP dipped slightly (1.00 -> 0.90), worth tracking but not the headline.

**Style transfer to nearby OOD.** referral_b (never trained) jumps 0.27 -> 0.47. The form of referral_a transfers to referral_b for free.

**Exposed: far-OOD degeneration on MSE.** Schema validity 1.00 -> 0.00, overlap 0.05 -> 0.01, `wrong_null_rate` 0.35 -> 0.77, `all_null_rate` 0.00 -> 0.60. The model emits almost nothing on a schema it never saw. This is the rational low-risk move given its training. 

## Run 3: model `v2.1-lite` on all four templates

**Model**: v2.1-lite, SFT on SOAP + referral_a + MSE. **Data**: in-distribution for SOAP / referral_a / MSE, zero-shot for referral_b. **Why**: test whether adding MSE training data cures the v2 degeneration on MSE, and what residual problems remain.

In [7]:
v2_1_summaries, v2_1_details = evaluate_model("v2.1")
print()
print_table(v2_1_summaries, columns=DISPLAY_COLUMNS)

# delta vs v2 on directly comparable templates only (skip mse because the split differs)
v2_lookup = {(r["template"], r["split"]): r for r in v2_summaries}
delta_rows = []
for r in v2_1_summaries:
    key = (r["template"], r["split"])
    base = v2_lookup.get(key)
    if base is None:
        continue  # mse: v2 split=zero_shot vs v2.1 split=in_dist -> not directly comparable
    delta_rows.append({
        "template":       r["template"],
        "split":          r["split"],
        "d_schema_valid": r["schema_valid_rate"]      - base["schema_valid_rate"],
        "d_overlap":      r["content_overlap_mean"]   - base["content_overlap_mean"],
        "d_grounding":    r["evidence_grounding_rate"]- base["evidence_grounding_rate"],
        "d_wrong_null":   r["wrong_null_rate"]        - base["wrong_null_rate"],
    })
print("\nDelta v2.1 - v2 (mse omitted because the eval split differs):")
print_table(delta_rows)

[run] v2.1 -> soap (in_dist, 10 rows) :: In-distribution. Check we did not regress SOAP after adding MSE.
[load] v2.1: v2_1_lite-qwen3b-multitemplate-q4_k_m.gguf


llama_context: n_ctx_seq (4096) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


[load] ready in 1.3s
  [ 1/10]    7371 ms  parse=1 schema=1 overlap=0.42 ground=1.00 all_null=0 wrong_null=1/8
  [ 2/10]    2978 ms  parse=1 schema=1 overlap=0.17 ground=1.00 all_null=0 wrong_null=3/7
  [ 3/10]    4223 ms  parse=1 schema=1 overlap=0.52 ground=1.00 all_null=0 wrong_null=0/6
  [ 4/10]    7340 ms  parse=1 schema=1 overlap=0.54 ground=1.00 all_null=0 wrong_null=3/13
  [ 5/10]    5726 ms  parse=1 schema=1 overlap=0.61 ground=1.00 all_null=0 wrong_null=0/10
  [ 6/10]    3703 ms  parse=1 schema=1 overlap=0.52 ground=1.00 all_null=0 wrong_null=1/7
  [ 7/10]    6103 ms  parse=1 schema=1 overlap=0.55 ground=1.00 all_null=0 wrong_null=1/12
  [ 8/10]    3610 ms  parse=1 schema=1 overlap=0.33 ground=1.00 all_null=0 wrong_null=2/5
  [ 9/10]    7417 ms  parse=1 schema=1 overlap=0.69 ground=0.89 all_null=0 wrong_null=1/10
  [10/10]    4046 ms  parse=1 schema=1 overlap=0.89 ground=1.00 all_null=0 wrong_null=0/8
[run] v2.1 -> referral_a (in_dist, 10 rows) :: In-distribution. Check refer

### Observation: v2.1-lite

v2.1-lite fixes v2's far-OOD degeneration and exposes the residual class that DPO is designed for.

**Fixed.** MSE on v2.1: schema 0.00 -> 1.00, overlap 0.01 -> 0.14, `wrong_null_rate` 0.77 -> 0.05. Adding MSE to SFT cures the degeneration. Diagnosis confirmed: v2 was under-trained, not broken.

**Held.** SOAP overlap 0.50 -> 0.52, referral_a 0.74 -> 0.70 (trivial regress), referral_b 0.47 -> 0.48.

**Exposed.** Two residuals remain.
1. referral_b is still untrained and still capped near 0.48. Problem 1 lives on for any template never seen.
2. On in-distribution templates, v2.1 still misses content and can emit ungrounded spans even when the JSON is schema-valid. These behavioural failures were present in v1 and v2 too. They were drowned out by louder problems. v2.1 makes them the dominant signal. The Problem 2 section below characterises them. DPO is the response.

## Run 4: model `v3` (DPO)

**Model**: v3, v2.1-lite further trained with DPO on verifier-scored preference pairs. In each pair the chosen output is a high-verifier-score extraction and the rejected output is a balanced draw across the failure buckets the verifier exposes (miss, over-populated, ungrounded span, schema-invalid) on the same transcript. **Data**: same eval as v2.1 (soap, referral_a, mse in-distribution, referral_b held out). **Why**: DPO targets Problem 2, the behavioural misses on trained templates that SFT's token loss cannot see. This run measures whether it moved.

DPO adds no template coverage, so referral_b stays a Problem 1 probe. We report whatever the numbers show.

In [8]:
v3_summaries, v3_details = evaluate_model("v3")
print()
print_table(v3_summaries, columns=DISPLAY_COLUMNS)

# delta vs v2.1 on the in-distribution templates DPO was trained to improve
v2_1_lookup = {(r["template"], r["split"]): r for r in v2_1_summaries}
delta_rows = []
for r in v3_summaries:
    base = v2_1_lookup.get((r["template"], r["split"]))
    if base is None:
        continue
    delta_rows.append({
        "template":       r["template"],
        "split":          r["split"],
        "d_schema_valid": r["schema_valid_rate"]       - base["schema_valid_rate"],
        "d_overlap":      r["content_overlap_mean"]    - base["content_overlap_mean"],
        "d_grounding":    r["evidence_grounding_rate"] - base["evidence_grounding_rate"],
        "d_wrong_null":   r["wrong_null_rate"]         - base["wrong_null_rate"],
    })
print("\nDelta v3 - v2.1 (positive d_overlap/d_schema/d_grounding = improvement; negative d_wrong_null = fewer misses):")
print_table(delta_rows)

[run] v3 -> soap (in_dist, 10 rows) :: In-distribution. Check DPO did not regress SOAP schema validity or content.
[load] v3: v3_qwen3b-multitemplate_dpo-q4_k_m.gguf


llama_context: n_ctx_seq (4096) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


[load] ready in 1.3s
  [ 1/10]    6903 ms  parse=1 schema=1 overlap=0.38 ground=1.00 all_null=0 wrong_null=1/8
  [ 2/10]    2971 ms  parse=1 schema=1 overlap=0.17 ground=1.00 all_null=0 wrong_null=3/7
  [ 3/10]    4464 ms  parse=1 schema=1 overlap=0.58 ground=1.00 all_null=0 wrong_null=0/6
  [ 4/10]    7552 ms  parse=0 schema=0 overlap=0.00 ground=1.00 all_null=1 wrong_null=13/13
  [ 5/10]    5740 ms  parse=1 schema=1 overlap=0.61 ground=1.00 all_null=0 wrong_null=0/10
  [ 6/10]    3694 ms  parse=1 schema=1 overlap=0.52 ground=1.00 all_null=0 wrong_null=1/7
  [ 7/10]    6088 ms  parse=1 schema=1 overlap=0.55 ground=1.00 all_null=0 wrong_null=1/12
  [ 8/10]    4412 ms  parse=1 schema=1 overlap=0.26 ground=1.00 all_null=0 wrong_null=2/5
  [ 9/10]    7227 ms  parse=1 schema=1 overlap=0.65 ground=1.00 all_null=0 wrong_null=1/10
  [10/10]    4045 ms  parse=1 schema=1 overlap=0.89 ground=1.00 all_null=0 wrong_null=0/8
[run] v3 -> referral_a (in_dist, 10 rows) :: In-distribution. A core targe

### Observation: v3 (DPO), 210 pairs, 2 epochs

Read against v2.1 on the in-distribution templates DPO was meant to improve. This is a mixed result, better behaved than the longer 4-epoch run, and it shows the method can work when it does not over-optimize.

**referral_a improved, the intended effect.** content_overlap 0.704 to 0.712, wrong_null_rate 0.072 to 0.058, ungrounded_span_rate 0.046 to 0.031. Misses down and hallucinations down at the same time, with schema_valid held at 1.00. This is exactly the Problem 2 movement DPO was built to produce, on the cleanest and best-structured of the trained templates.

**soap regressed, but through a single collapse.** schema_valid 1.00 to 0.90, content_overlap 0.52 to 0.46, wrong_null_rate 0.14 to 0.26, all_null_rate 0.00 to 0.10. Only one of ten outputs broke, index 4, which became unparseable all-null and missed all 13 of its gold fields. That single item is the richest-gold SOAP transcript in the set. Remove it and SOAP's wrong_null_rate is about 0.12, level with v2.1. The damage is one degenerate generation on the longest target, not a systemic SOAP failure.

**mse drifted the wrong way.** content_overlap 0.137 to 0.148 (marginal gain), but wrong_null_rate 0.046 to 0.091 and ungrounded_span_rate 0.063 to 0.119 both rose. More misses and more invented spans, no schema cost.

**referral_b stretch hypothesis did not hold.** content_overlap 0.482 to 0.482 (flat), and grounding fell (ungrounded_span_rate 0.053 to 0.105). No positive carry-over from referral_a, as expected for an unseen ontology.

**Diagnosis.** Two epochs on 210 pairs is closer to the right amount of DPO than the 4-epoch run was. It produced a genuine, if small, gain on referral_a instead of the broad collapse the longer run caused. The remaining failures are still the offline-DPO signature. The single SOAP collapse is the longest target degenerating under preference pressure, a likelihood displacement on a transcript whose valid output shares early tokens with the rejected nulls. The mse drift shows the signal is too weak to reduce misses on the lowest-overlap template without also raising hallucinations. The takeaway is that DPO here is real but fragile and uneven, and highly sensitive to how long it trains.

## Combined progression: v1, v2, v2.1, v3 in one place

This is the single table that shows the full story numerically, now including the DPO checkpoint.

In [9]:
all_summaries = v1_summaries + v2_summaries + v2_1_summaries + v3_summaries
all_details = v1_details + v2_details + v2_1_details + v3_details
summary_lookup = {(r["model"], r["template"]): r for r in all_summaries}

rows = []
for template_name in TEMPLATE_ORDER:
    for model_name in MODEL_ORDER:
        r = summary_lookup.get((model_name, template_name))
        if r is None:
            continue
        rows.append({
            "template": template_name,
            "model": model_name,
            "split": r["split"],
            "schema_valid":   r["schema_valid_rate"],
            "overlap":        r["content_overlap_mean"],
            "grounding":      r["evidence_grounding_rate"],
            "ungrounded":     r["ungrounded_span_rate"],
            "wrong_null":     r["wrong_null_rate"],
            "correct_null":   r["correct_null_rate"],
            "all_null":       r["all_null_rate"],
        })
print_table(rows, columns=[
    "template", "model", "split",
    "schema_valid", "overlap", "grounding", "ungrounded",
    "wrong_null", "correct_null", "all_null",
])

template   | model | split     | schema_valid | overlap | grounding | ungrounded | wrong_null | correct_null | all_null
-----------+-------+-----------+--------------+---------+-----------+------------+------------+--------------+---------
soap       | v1    | in_dist   | 1.0000       | 0.3404  | 0.9538    | 0.0462     | 0.2558     | 1.0000       | 0.0000  
soap       | v2    | in_dist   | 0.9000       | 0.5032  | 0.9512    | 0.0488     | 0.1628     | 1.0000       | 0.0000  
soap       | v2.1  | in_dist   | 1.0000       | 0.5239  | 0.9875    | 0.0125     | 0.1395     | 1.0000       | 0.0000  
soap       | v3    | in_dist   | 0.9000       | 0.4612  | 1.0000    | 0.0000     | 0.2558     | 1.0000       | 0.1000  
referral_a | v1    | in_dist   | 1.0000       | 0.5037  | 0.8485    | 0.1515     | 0.0870     | 1.0000       | 0.0000  
referral_a | v2    | in_dist   | 1.0000       | 0.7400  | 0.9846    | 0.0154     | 0.0725     | 1.0000       | 0.0000  
referral_a | v2.1  | in_dist   | 1.0000 

## Problem 1: SFT is bounded by training-set coverage

This is the limit no preference method can lift. The fix is data.

In [10]:
problem1_rows = []
for model_name in MODEL_ORDER:
    r = summary_lookup.get((model_name, "referral_b"))
    if r:
        problem1_rows.append({
            "probe": "referral_b (unseen by all)",
            "model": model_name,
            "split": r["split"],
            "schema_valid": r["schema_valid_rate"],
            "overlap":      r["content_overlap_mean"],
            "grounding":    r["evidence_grounding_rate"],
            "wrong_null":   r["wrong_null_rate"],
        })
for model_name in ["v1", "v2"]:
    r = summary_lookup.get((model_name, "mse"))
    if r:
        problem1_rows.append({
            "probe": "mse (zero-shot for v1/v2)",
            "model": model_name,
            "split": r["split"],
            "schema_valid": r["schema_valid_rate"],
            "overlap":      r["content_overlap_mean"],
            "grounding":    r["evidence_grounding_rate"],
            "wrong_null":   r["wrong_null_rate"],
        })
print_table(problem1_rows)

probe                      | model | split     | schema_valid | overlap | grounding | wrong_null
---------------------------+-------+-----------+--------------+---------+-----------+-----------
referral_b (unseen by all) | v1    | held_out  | 1.0000       | 0.2651  | 0.8364    | 0.1923    
referral_b (unseen by all) | v2    | held_out  | 1.0000       | 0.4678  | 0.9153    | 0.0385    
referral_b (unseen by all) | v2.1  | held_out  | 1.0000       | 0.4816  | 0.9474    | 0.0192    
referral_b (unseen by all) | v3    | held_out  | 1.0000       | 0.4825  | 0.8947    | 0.0385    
mse (zero-shot for v1/v2)  | v1    | zero_shot | 1.0000       | 0.0546  | 0.9149    | 0.3542    
mse (zero-shot for v1/v2)  | v2    | zero_shot | 0.0000       | 0.0104  | 1.0000    | 0.7708    


### Observation: Problem 1

referral_b zero-shot content overlap across the progression: v1 0.27, v2 0.47, v2.1 0.48. In-distribution referral_a on v2.1 is 0.70.

The 0.27 -> 0.47 jump (v1 to v2) is style transfer from training referral_a, since referral_b is structurally similar. The 0.47 -> 0.48 step (v2 to v2.1) is essentially zero. Adding MSE data did not help referral_b. Zero-shot referral_b stays about 22 points below in-distribution referral_a on the same model.

This is the SFT coverage ceiling. SFT cannot make a model good at an ontology it has never seen. DPO and GRPO will not close this gap either. They reshape behaviour over fields the model already knows about. The fix for Problem 1 is more training coverage, or retrieval-style schema injection, or schema-conditioned synthetic data. It is a data problem, not a method problem.

v3 (DPO) sits in the referral_b rows of the combined table at overlap 0.482, identical to v2.1's 0.482 and with worse grounding (ungrounded_span_rate 0.053 to 0.105). The stretch hypothesis did not hold. Preference learning on the trained templates produced no positive carry-over to the unseen referral_b ontology. This is the expected outcome. DPO reshapes behaviour over fields the model already knows, it does not add an unseen ontology. Problem 1 stays a data problem, and no preference method will close it.

The MSE-on-v2 degeneration is the same phenomenon in a more extreme form. The next two cells show it qualitatively.

### Show, don't just tell: what MSE-on-v2 looks like qualitatively

The table above gives the numbers. The diff below shows what those numbers mean on real transcripts. v2 (which never saw MSE) returns nulls. v2.1-lite (which did) populates the fields. This is one more illustration of Problem 1, not a new problem.

In [11]:
def _pretty(obj):
    if obj is None:
        return "<no parseable output>"
    try:
        return json.dumps(obj, indent=2, ensure_ascii=False)
    except Exception:
        return str(obj)


def render_diff(transcript, gold, pred_map, title=""):
    cols = list(pred_map.keys())
    header_cells = "".join(f"<th style='width:{int(66/len(cols))}%;text-align:left;padding:6px;background:#f0f0f0'>{html.escape(c)}</th>" for c in cols)
    header = (
        "<tr>"
        "<th style='width:34%;text-align:left;padding:6px;background:#e8f4e8'>gold</th>"
        f"{header_cells}"
        "</tr>"
    )
    def cell(text):
        return (
            "<td style='vertical-align:top;padding:6px;border:1px solid #ccc'>"
            f"<pre style='white-space:pre-wrap;font-size:12px;margin:0'>{html.escape(text)}</pre>"
            "</td>"
        )
    body = "<tr>" + cell(_pretty(gold)) + "".join(cell(_pretty(pred_map[c])) for c in cols) + "</tr>"
    table = f"<table style='width:100%;border-collapse:collapse;margin-top:6px'>{header}{body}</table>"
    transcript_block = (
        "<details style='margin-top:10px'><summary><b>Transcript</b> (click to expand)</summary>"
        f"<pre style='white-space:pre-wrap;font-size:12px;background:#fafafa;padding:8px;border:1px solid #ddd'>{html.escape(transcript)}</pre>"
        "</details>"
    )
    title_block = f"<h4 style='margin-bottom:4px'>{html.escape(title)}</h4>" if title else ""
    display(HTML(title_block + transcript_block + table))


def pick_examples(details, template, model, n=3, prefer="gold_filled"):
    rows = [d for d in details if d["template"] == template and d["model"] == model]
    if prefer == "gold_filled":
        rows.sort(key=lambda d: -d["gold_filled"])
    return rows[:n]


def pred_for(details, transcript, model):
    for d in details:
        if d["model"] == model and d["transcript"] == transcript:
            return d["pred_label"]
    return None

In [12]:
# Pick the 3 MSE transcripts with the richest gold (most filled fields) from v2.1's in-dist MSE run.
examples = pick_examples(v2_1_details, template="mse", model="v2.1", n=3)
for i, ex in enumerate(examples, 1):
    title = f"MSE example {i}/{len(examples)}: gold filled fields: {ex['gold_filled']}, gold null fields: {ex['gold_null']}"
    pred_map = {
        "v2 (zero-shot mse)": pred_for(v2_details, ex["transcript"], "v2"),
        "v2.1 (in-dist mse)": ex["pred_label"],
    }
    # v2 used a different eval file for mse; transcripts will not match. Fall back to a v2 sample on the same row index if needed.
    if pred_map["v2 (zero-shot mse)"] is None:
        # take the i-th v2 mse output as illustrative; mark as different transcript
        v2_mse = [d for d in v2_details if d["template"] == "mse"]
        if i - 1 < len(v2_mse):
            illustrative = v2_mse[i - 1]
            title += "  [v2 column shown from a different MSE transcript: v2 and v2.1 use different MSE eval files. See the v2 vs v2.1 MSE rows in the combined progression table for aggregate numbers.]"
            pred_map["v2 (zero-shot mse)"] = illustrative["pred_label"]
    render_diff(ex["transcript"], ex["gold"], pred_map, title=title)

gold,v2 (zero-shot mse),v2.1 (in-dist mse)
"{ ""appearance"": { ""text"": null, ""evidence_span"": null }, ""behaviour"": { ""text"": ""repetitive checking, psychomotor agitation"", ""evidence_span"": ""i can't stop checking the locks on my front door"" }, ""speech"": { ""text"": ""rapid speech"", ""evidence_span"": ""i notice you're speaking quite fast"" }, ""mood"": { ""text"": ""anxious, scared"", ""evidence_span"": ""it's just the anxiety again, it's been getting worse lately"" }, ""affect"": { ""text"": ""concerned, anxious"", ""evidence_span"": ""looking around a lot"" }, ""thought"": { ""text"": ""catastrophizing, intrusive thoughts"", ""evidence_span"": ""like i have to do something right now or something bad will happen"" }, ""cognition"": { ""text"": ""poor concentration, sleep disturbance"", ""evidence_span"": ""i can't concentrate at work, i keep making mistakes, and i haven't slept in three nights"" }, ""insight"": { ""text"": ""partial insight — recognizes symptoms as panic"", ""evidence_span"": ""but i know i should calm down, it's just the panic again"" } }","{ ""appearance"": null, ""behaviour"": { ""text"": ""talking fast and fidgeting"", ""evidence_span"": ""i think i'm just talking fast and fidgeting"" }, ""speech"": null, ""mood"": { ""text"": ""anxious"", ""evidence_span"": ""just feeling really anxious all the time"" }, ""affect"": null, ""thought"": { ""text"": ""paranoid about people watching me"", ""evidence_span"": ""i'm just really paranoid about people watching me"" }, ""cognition"": null, ""insight"": { ""text"": ""need help but scared to mess it up"", ""evidence_span"": ""i know i need help, but i'm scared i'll just mess it up again"" } }","{ ""appearance"": { ""text"": null, ""evidence_span"": null }, ""behaviour"": { ""text"": ""on edge, looking around a lot"", ""evidence_span"": ""i notice you're speaking quite fast and looking around a lot, is that how you feel today"" }, ""speech"": { ""text"": ""speaking quite fast and looking around a lot"", ""evidence_span"": ""i notice you're speaking quite fast and looking around a lot, is that how you feel today"" }, ""mood"": { ""text"": ""on edge, feeling scared"", ""evidence_span"": ""i feel on edge, like i have to do something right now or something bad will happen"" }, ""affect"": { ""text"": null, ""evidence_span"": null }, ""thought"": { ""text"": ""racing thoughts, fear of bad things happening"", ""evidence_span"": ""i can't stop checking the locks on my front door, maybe ten times a day"" }, ""cognition"": { ""text"": ""poor concentration, sleep disturbance"", ""evidence_span"": ""i can't concentrate at work, i keep making mistakes, and i haven't slept in three nights because i'm just too worried"" }, ""insight"": { ""text"": ""partial insight, acknowledges fear is irrational"", ""evidence_span"": ""i know i should calm down, it's just the panic again"" } }"


gold,v2 (zero-shot mse),v2.1 (in-dist mse)
"{ ""appearance"": { ""text"": null, ""evidence_span"": null }, ""behaviour"": { ""text"": ""restless, unable to focus"", ""evidence_span"": ""can't stop worrying about emails"" }, ""speech"": { ""text"": null, ""evidence_span"": null }, ""mood"": { ""text"": ""overwhelmed, anxious"", ""evidence_span"": ""feeling really anxious about work"" }, ""affect"": { ""text"": null, ""evidence_span"": null }, ""thought"": { ""text"": ""ruminative, preoccupied with work mistakes"", ""evidence_span"": ""just replaying every mistake i made"" }, ""cognition"": { ""text"": ""poor concentration, sleep disturbance"", ""evidence_span"": ""can't focus on anything"" }, ""insight"": { ""text"": ""partial insight — acknowledges need for help but doubtful of outcome"", ""evidence_span"": ""i know i should see someone, but i'm not sure if i'll get better"" } }","{ ""appearance"": null, ""behaviour"": null, ""speech"": ""feeling really anxious"", ""mood"": null, ""affect"": null, ""thought"": ""voices saying i'm a failure and everyone hates me"", ""cognition"": null, ""insight"": ""know it's just stress"" }","{ ""appearance"": { ""text"": null, ""evidence_span"": null }, ""behaviour"": { ""text"": ""psychomotor agitation, tremor"", ""evidence_span"": ""my hands are trembling and i can't focus on anything"" }, ""speech"": { ""text"": ""speech pressured, anxious"", ""evidence_span"": ""i've been feeling really anxious about work, can't stop worrying about emails"" }, ""mood"": { ""text"": ""anxious, overwhelmed"", ""evidence_span"": ""i've been feeling really anxious about work, can't stop worrying about emails"" }, ""affect"": { ""text"": null, ""evidence_span"": null }, ""thought"": { ""text"": ""intrusive worry, racing thoughts"", ""evidence_span"": ""i find myself awake for hours, just replaying every mistake i made"" }, ""cognition"": { ""text"": ""poor concentration, fatigue"", ""evidence_span"": ""i find myself awake for hours, just replaying every mistake i made"" }, ""insight"": { ""text"": ""partial insight, acknowledges need for support"", ""evidence_span"": ""i know i should see someone, but i'm not sure if i'll get better"" } }"


gold,v2 (zero-shot mse),v2.1 (in-dist mse)
"{ ""appearance"": { ""text"": null, ""evidence_span"": null }, ""behaviour"": { ""text"": ""repetitive checking behaviour"", ""evidence_span"": ""i can't stop checking the locks"" }, ""speech"": { ""text"": null, ""evidence_span"": null }, ""mood"": { ""text"": ""anxious, fearful"", ""evidence_span"": ""feeling really on edge"" }, ""affect"": { ""text"": null, ""evidence_span"": null }, ""thought"": { ""text"": ""intrusive fears, no SI"", ""evidence_span"": ""scared of the panic attacks"" }, ""cognition"": { ""text"": ""poor concentration, sleep disturbance"", ""evidence_span"": ""i can barely read a page, everything feels foggy"" }, ""insight"": { ""text"": ""good insight"", ""evidence_span"": ""i know i need to see someone"" } }","{ ""appearance"": null, ""behaviour"": null, ""speech"": { ""text"": ""i've been really anxious about work, can't stop worrying about mistakes"", ""evidence_span"": ""i've been really anxious about work, can't stop worrying about mistakes"" }, ""mood"": null, ""affect"": null, ""thought"": { ""text"": ""my heart races and i get sweaty palms every time i think about the meeting"", ""evidence_span"": ""my heart races and i get sweaty palms every time i think about the meeting"" }, ""cognition"": null, ""insight"": null }","{ ""appearance"": { ""text"": null, ""evidence_span"": null }, ""behaviour"": { ""text"": ""psychomotor agitation, avoidance behaviour"", ""evidence_span"": ""my heart is racing, i can't stop checking the locks, it's so exhausting"" }, ""speech"": { ""text"": ""racing speech, pressured speech"", ""evidence_span"": ""my heart is racing, i can't stop checking the locks, it's so exhausting"" }, ""mood"": { ""text"": ""anxious, fearful"", ""evidence_span"": ""feeling really on edge"" }, ""affect"": { ""text"": ""anxious, fearful"", ""evidence_span"": ""feeling really on edge"" }, ""thought"": { ""text"": ""intrusive thoughts, fear of panic attacks"", ""evidence_span"": ""i know i need to see someone"" }, ""cognition"": { ""text"": ""concentration difficulties, sleep disturbance"", ""evidence_span"": ""i wake up every hour, i feel like i'm going crazy"" }, ""insight"": { ""text"": ""partial insight, acknowledges need for help"", ""evidence_span"": ""yeah, mostly anxiety again, feeling really on edge"" } }"


### Observation: MSE side-by-side

On the same three MSE transcripts, gold has populated fields across `appearance`, `behaviour`, `speech`, `mood_affect`, `thought_process`, `perception`, `cognition`, `insight_judgement`, `risk`. v2 returns nulls in almost every slot. v2.1 populates the same fields with reasonable content.

The cure was training data. This is Problem 1 made concrete.

## Problem 2: behavioural misses on in-distribution data

v2.1 has the schema. It still misses content and occasionally emits ungrounded spans. These outputs are valid JSON, so SFT's token loss had no signal to punish them. This is the class DPO targets.

In [13]:
# Filter v2.1 details to in-distribution templates only.
# These are templates v2.1 was trained on (soap, referral_a, mse).
# referral_b is excluded because it is Problem 1 territory.

IN_DIST = ["soap", "referral_a", "mse"]
v2_1_in_dist = [d for d in v2_1_details if d["template"] in IN_DIST]

from collections import defaultdict
buckets = defaultdict(list)
for d in v2_1_in_dist:
    buckets[d["template"]].append(d)

def mean(xs):
    return sum(xs) / len(xs) if xs else 0.0

print("| template | n | schema_valid | content_overlap | wrong_null_rate | ungrounded_span_rate |")
print("|---|---|---|---|---|---|")
for tpl in IN_DIST:
    ds = buckets[tpl]
    grounded = sum(d["grounded_spans"] for d in ds)
    total = sum(d["total_spans"] for d in ds)
    gold_filled = sum(d["gold_filled"] for d in ds)
    wn = sum(d["wrong_null"] for d in ds)
    ugr = 1.0 - (grounded / total) if total else 0.0
    wnr = (wn / gold_filled) if gold_filled else 0.0
    print(f"| {tpl} | {len(ds)} | {mean([d['schema_valid'] for d in ds]):.2f} | {mean([d['content_overlap'] for d in ds]):.2f} | {wnr:.2f} | {ugr:.2f} |")


| template | n | schema_valid | content_overlap | wrong_null_rate | ungrounded_span_rate |
|---|---|---|---|---|---|
| soap | 10 | 1.00 | 0.52 | 0.14 | 0.01 |
| referral_a | 10 | 1.00 | 0.70 | 0.07 | 0.05 |
| mse | 10 | 1.00 | 0.14 | 0.05 | 0.06 |


### Reading the table

| template | n | schema_valid | content_overlap | wrong_null_rate | ungrounded_span_rate |
|---|---|---|---|---|---|
| soap | 10 | 1.00 | 0.52 | 0.14 | 0.01 |
| referral_a | 10 | 1.00 | 0.70 | 0.07 | 0.05 |
| mse | 10 | 1.00 | 0.14 | 0.05 | 0.06 |

`schema_valid` is solved on every in-distribution template. `content_overlap` is not. Even on SOAP, the template v2.1 has seen the most, the model leaves 14 percent of gold-filled fields null (`wrong_null_rate` 0.14). On MSE, content overlap is only 0.14 despite the model having been trained on the schema. Ungrounded spans appear on referral_a (0.05) and MSE (0.06), so the model is also occasionally inventing evidence.

Token cross-entropy treats a schema-valid null and the correct evidence-grounded answer as two valid token sequences. Both are well-formed JSON. SFT cannot distinguish them. Whole-output preference (DPO) can, because it scores the full output against a verifier.

In [14]:
# Pick concrete v2.1 in-distribution failure examples covering distinct sub-modes:
#   A) schema_valid=1 with wrong_null > 0  (content miss on trained schema)
#   B) schema_valid=1 with ungrounded spans (hallucination on trained schema)
#   C) schema_valid=1 with low content_overlap (content-blind extraction)
# We deduplicate and cap at 3 examples.

def _pick(cands, picked_ids):
    for d in cands:
        key = (d["template"], d["index"])
        if key not in picked_ids:
            picked_ids.add(key)
            return d
    return None

picked = []
picked_ids = set()

a_cands = sorted(
    [d for d in v2_1_in_dist if d["schema_valid"] == 1 and d["wrong_null"] > 0],
    key=lambda d: -d["wrong_null"],
)
a = _pick(a_cands, picked_ids)
if a: picked.append(("A: content miss on trained schema", a))

b_cands = sorted(
    [d for d in v2_1_in_dist if d["schema_valid"] == 1 and d["total_spans"] > 0 and d["grounded_spans"] < d["total_spans"]],
    key=lambda d: -(d["total_spans"] - d["grounded_spans"]),
)
b = _pick(b_cands, picked_ids)
if b: picked.append(("B: ungrounded span on trained schema", b))

c_cands = sorted(
    [d for d in v2_1_in_dist if d["schema_valid"] == 1 and d["content_overlap"] < 0.4],
    key=lambda d: d["content_overlap"],
)
c = _pick(c_cands, picked_ids)
if c: picked.append(("C: content-blind extraction", c))

print(f"Found {len(picked)} of 3 sub-modes on v2.1 in-distribution data.")
for label, d in picked:
    print(f"  - {label}: template={d['template']} index={d['index']} schema_valid={d['schema_valid']} overlap={d['content_overlap']:.2f} wrong_null={d['wrong_null']}/{d['gold_filled']} ungrounded_spans={d['total_spans']-d['grounded_spans']}/{d['total_spans']}")


Found 3 of 3 sub-modes on v2.1 in-distribution data.
  - A: content miss on trained schema: template=soap index=2 schema_valid=1 overlap=0.17 wrong_null=3/7 ungrounded_spans=0/5
  - B: ungrounded span on trained schema: template=mse index=5 schema_valid=1 overlap=0.20 wrong_null=0/5 ungrounded_spans=2/5
  - C: content-blind extraction: template=mse index=2 schema_valid=1 overlap=0.00 wrong_null=0/3 ungrounded_spans=0/3


In [15]:
# Render the picked examples as side-by-side HTML diffs (transcript / gold / v2.1).
# Reuses the _pretty helper defined earlier in the Problem 1 section.

from IPython.display import HTML, display

def _render(label, d):
    return f'''
<div style="margin-bottom:18px">
  <h4 style="margin:0 0 6px 0">{html.escape(label)} &middot; template=<code>{html.escape(d['template'])}</code> &middot; index={d['index']}</h4>
  <table style="width:100%; border-collapse:collapse; font-family:monospace; font-size:12px">
    <tr style="background:#eef">
      <th style="width:34%; text-align:left; padding:4px; border:1px solid #ccc">transcript (truncated)</th>
      <th style="width:33%; text-align:left; padding:4px; border:1px solid #ccc">gold</th>
      <th style="width:33%; text-align:left; padding:4px; border:1px solid #ccc">v2.1 prediction</th>
    </tr>
    <tr style="vertical-align:top">
      <td style="padding:4px; border:1px solid #ccc; white-space:pre-wrap">{html.escape(d['transcript'][:900])}{'...' if len(d['transcript'])>900 else ''}</td>
      <td style="padding:4px; border:1px solid #ccc; white-space:pre-wrap">{html.escape(_pretty(d['gold']))}</td>
      <td style="padding:4px; border:1px solid #ccc; white-space:pre-wrap">{html.escape(_pretty(d['pred_label']))}</td>
    </tr>
  </table>
</div>
'''

display(HTML("".join(_render(label, d) for label, d in picked)))


transcript (truncated),gold,v2.1 prediction
"HCP: hi there, whats the matter? Patient: so i woke up with a terrible headache today HCP: how long has this been going on? Patient: just this morning, really sharp HCP: any history of migraines or high blood pressure? Patient: no history of migraines but my BP was a bit high last time i checked HCP: ok, lets check your BP, looks like 145 over 90 today Patient: yep that feels about right HCP: sounds like tension headache maybe, try some panadol and rest Patient: thanks doc","{ ""subjective"": { ""chief_complaint"": { ""text"": ""headache"", ""evidence_span"": ""terrible headache"" }, ""hpi"": { ""text"": ""headache started this morning, sharp pain"", ""evidence_span"": ""woke up with a terrible headache today"" }, ""pmh"": [ { ""text"": ""high blood pressure"", ""evidence_span"": ""high blood pressure"" } ], ""meds"": [], ""allergies"": [] }, ""objective"": { ""exam"": { ""text"": ""BP 145 over 90"", ""evidence_span"": ""looks like 145 over 90 today"" } }, ""assessment"": { ""problems"": [ { ""name"": ""tension headache"", ""evidence_span"": ""tension headache"" } ] }, ""plan"": { ""items"": [ { ""action"": ""take panadol"", ""evidence_span"": ""try some panadol"" }, { ""action"": ""rest"", ""evidence_span"": ""rest"" } ] } }","{ ""subjective"": { ""chief_complaint"": { ""text"": ""headache"", ""evidence_span"": ""headache"" }, ""hpi"": { ""text"": ""woke up with a terrible headache today"", ""evidence_span"": ""woke up with a terrible headache today"" }, ""pmh"": [], ""meds"": [ { ""text"": ""panadol"", ""evidence_span"": ""try some panadol"" } ], ""allergies"": [] }, ""objective"": { ""exam"": { ""text"": null, ""evidence_span"": null } }, ""assessment"": { ""problems"": [ { ""name"": ""tension headache"", ""evidence_span"": ""sounds like tension headache"" } ] }, ""plan"": { ""items"": [ { ""action"": ""rest"", ""evidence_span"": ""rest"" } ] } }"
transcript (truncated),gold,v2.1 prediction
"HCP: hello, i'm dr lee, how are you doing today Patient: hi dr lee, yeah i'm okay i suppose, just really tired all the time HCP: sounds like you've had a rough few weeks, what brings you in today Patient: well mostly the anxiety, i feel like i can't breathe when i'm in public HCP: that sounds really frightening, how long has this been going on Patient: probably about three months now, since my mum got sick, i just get panicky HCP: and has this affected your daily life at work or home Patient: yes, i can't leave the house much, i'm avoiding the shops and the bus stop HCP: i understand that's hard, have you noticed any changes in your sleep or appetite Patient: not really, i eat okay but i'm definitely not sleeping properly, waking up at 3am HCP: ok, and do you feel this is related to your mother's condition Patient: yeah i think so, i'm just scared something bad will happen to her HCP: do...","{ ""appearance"": { ""text"": null, ""evidence_span"": null }, ""behaviour"": { ""text"": ""social avoidance, agitated anxiety"", ""evidence_span"": ""i can't leave the house much, i'm avoiding the shops and the bus stop"" }, ""speech"": { ""text"": null, ""evidence_span"": null }, ""mood"": { ""text"": ""fearful, anxious"", ""evidence_span"": ""i feel like i can't breathe when i'm in public"" }, ""affect"": { ""text"": null, ""evidence_span"": null }, ""thought"": { ""text"": ""excessive worry, fear of harm to mother"", ""evidence_span"": ""i'm just scared something bad will happen to her"" }, ""cognition"": { ""text"": ""poor sleep, impaired concentration"", ""evidence_span"": ""i'm definitely not sleeping properly, waking up at 3am"" }, ""insight"": { ""text"": ""partial insight, uncertain about treatment"", ""evidence_span"": ""i'm not sure what you want me to do though"" } }","{ ""appearance"": { ""text"": null, ""evidence_span"": null }, ""behaviour"": { ""text"": ""avoiding the shops and the bus stop"", ""evidence_span"": ""i can't leave the house much, i'm avoiding the shops and the bus stop"" },

### Observation: Problem 2

All three failure sub-modes are present on v2.1 in-distribution data.

| sub-mode | example | what went wrong |
|---|---|---|
| A: content miss on trained schema | soap, index 2, overlap 0.17, wrong_null 3 of 7 | Schema-valid SOAP output that left 3 of 7 gold-populated fields null on a transcript SOAP was trained on. |
| B: ungrounded span on trained schema | mse, index 5, overlap 0.20, 2 of 5 spans ungrounded | Two evidence_span strings did not appear verbatim in the transcript. The model invented them. |
| C: content-blind extraction | mse, index 2, overlap 0.00, schema_valid 1 | JSON parsed, schema matched, content overlap with gold was zero. The output had the right shape and the wrong content. |

These are all in-distribution. The JSON is well-formed. Token cross-entropy in SFT cannot punish them because the wrong output and the right output are both valid token sequences for a trained schema. There is no gradient that says one is preferred.

DPO closes that gap. Given a pair (chosen, rejected) where `chosen` is a high-verifier-score extraction and `rejected` is a low-verifier-score extraction on the same transcript, DPO updates the policy to prefer chosen over rejected. The verifier already exposes three scoring terms (schema_valid, content_overlap, evidence grounding) so we can build preference pairs without training a separate reward model.

Secondary hypothesis worth testing after DPO trains: because referral_b shares structure with referral_a, preference learning on in-distribution behavioural failures may also nudge referral_b further than SFT did. We will not assume this works. We will measure it.

## Problem 2 revisited: did DPO (v3) move the behavioural metrics?

The section above characterised v2.1's Problem 2 failures. v3 is the DPO response. Here we put v2.1 and v3 side by side on the in-distribution templates only (soap, referral_a, mse). referral_b is excluded because it is Problem 1 territory.

The success criteria, set before looking at the numbers:

- `wrong_null_rate` should fall (fewer misses).
- `ungrounded_span_rate` should not rise much (no miss-to-hallucination trade).
- `content_overlap_mean` should rise or hold.
- `schema_valid_rate` should stay at 1.00.

In [16]:
# v2.1 vs v3 on in-distribution templates, computed from per-item details.
IN_DIST = ["soap", "referral_a", "mse"]

def _mean(xs):
    return sum(xs) / len(xs) if xs else 0.0

def in_dist_stats(details, template):
    ds = [d for d in details if d["template"] == template]
    grounded = sum(d["grounded_spans"] for d in ds)
    total = sum(d["total_spans"] for d in ds)
    gold_filled = sum(d["gold_filled"] for d in ds)
    wn = sum(d["wrong_null"] for d in ds)
    return {
        "n": len(ds),
        "schema_valid": _mean([d["schema_valid"] for d in ds]),
        "content_overlap": _mean([d["content_overlap"] for d in ds]),
        "wrong_null_rate": (wn / gold_filled) if gold_filled else 0.0,
        "ungrounded_span_rate": (1.0 - grounded / total) if total else 0.0,
    }

print("| template | model | n | schema_valid | content_overlap | wrong_null_rate | ungrounded_span_rate |")
print("|---|---|---|---|---|---|---|")
for tpl in IN_DIST:
    a = in_dist_stats(v2_1_details, tpl)
    b = in_dist_stats(v3_details, tpl)
    print(f"| {tpl} | v2.1 | {a['n']} | {a['schema_valid']:.2f} | {a['content_overlap']:.2f} | {a['wrong_null_rate']:.2f} | {a['ungrounded_span_rate']:.2f} |")
    print(f"| {tpl} | v3   | {b['n']} | {b['schema_valid']:.2f} | {b['content_overlap']:.2f} | {b['wrong_null_rate']:.2f} | {b['ungrounded_span_rate']:.2f} |")

| template | model | n | schema_valid | content_overlap | wrong_null_rate | ungrounded_span_rate |
|---|---|---|---|---|---|---|
| soap | v2.1 | 10 | 1.00 | 0.52 | 0.14 | 0.01 |
| soap | v3   | 10 | 0.90 | 0.46 | 0.26 | 0.00 |
| referral_a | v2.1 | 10 | 1.00 | 0.70 | 0.07 | 0.05 |
| referral_a | v3   | 10 | 1.00 | 0.71 | 0.06 | 0.03 |
| mse | v2.1 | 10 | 1.00 | 0.14 | 0.05 | 0.06 |
| mse | v3   | 10 | 1.00 | 0.15 | 0.09 | 0.12 |


In [17]:
# Same transcripts picked as v2.1 Problem 2 failures, now with the v3 (DPO) column.
# If DPO worked, v3 fills fields v2.1 left null and drops invented spans on these exact rows.

def _pred_for_v3(transcript):
    for d in v3_details:
        if d["transcript"] == transcript:
            return d["pred_label"]
    return None

def _render3(label, d):
    v3_pred = _pred_for_v3(d["transcript"])
    return f'''
<div style="margin-bottom:18px">
  <h4 style="margin:0 0 6px 0">{html.escape(label)} &middot; template=<code>{html.escape(d['template'])}</code> &middot; index={d['index']}</h4>
  <table style="width:100%; border-collapse:collapse; font-family:monospace; font-size:12px">
    <tr style="background:#eef">
      <th style="width:25%; text-align:left; padding:4px; border:1px solid #ccc">transcript (truncated)</th>
      <th style="width:25%; text-align:left; padding:4px; border:1px solid #ccc">gold</th>
      <th style="width:25%; text-align:left; padding:4px; border:1px solid #ccc">v2.1 prediction</th>
      <th style="width:25%; text-align:left; padding:4px; border:1px solid #ccc">v3 (DPO) prediction</th>
    </tr>
    <tr style="vertical-align:top">
      <td style="padding:4px; border:1px solid #ccc; white-space:pre-wrap">{html.escape(d['transcript'][:900])}{'...' if len(d['transcript'])>900 else ''}</td>
      <td style="padding:4px; border:1px solid #ccc; white-space:pre-wrap">{html.escape(_pretty(d['gold']))}</td>
      <td style="padding:4px; border:1px solid #ccc; white-space:pre-wrap">{html.escape(_pretty(d['pred_label']))}</td>
      <td style="padding:4px; border:1px solid #ccc; white-space:pre-wrap">{html.escape(_pretty(v3_pred))}</td>
    </tr>
  </table>
</div>
'''

display(HTML("".join(_render3(label, d) for label, d in picked)))

transcript (truncated),gold,v2.1 prediction,v3 (DPO) prediction
"HCP: hi there, whats the matter? Patient: so i woke up with a terrible headache today HCP: how long has this been going on? Patient: just this morning, really sharp HCP: any history of migraines or high blood pressure? Patient: no history of migraines but my BP was a bit high last time i checked HCP: ok, lets check your BP, looks like 145 over 90 today Patient: yep that feels about right HCP: sounds like tension headache maybe, try some panadol and rest Patient: thanks doc","{ ""subjective"": { ""chief_complaint"": { ""text"": ""headache"", ""evidence_span"": ""terrible headache"" }, ""hpi"": { ""text"": ""headache started this morning, sharp pain"", ""evidence_span"": ""woke up with a terrible headache today"" }, ""pmh"": [ { ""text"": ""high blood pressure"", ""evidence_span"": ""high blood pressure"" } ], ""meds"": [], ""allergies"": [] }, ""objective"": { ""exam"": { ""text"": ""BP 145 over 90"", ""evidence_span"": ""looks like 145 over 90 today"" } }, ""assessment"": { ""problems"": [ { ""name"": ""tension headache"", ""evidence_span"": ""tension headache"" } ] }, ""plan"": { ""items"": [ { ""action"": ""take panadol"", ""evidence_span"": ""try some panadol"" }, { ""action"": ""rest"", ""evidence_span"": ""rest"" } ] } }","{ ""subjective"": { ""chief_complaint"": { ""text"": ""headache"", ""evidence_span"": ""headache"" }, ""hpi"": { ""text"": ""woke up with a terrible headache today"", ""evidence_span"": ""woke up with a terrible headache today"" }, ""pmh"": [], ""meds"": [ { ""text"": ""panadol"", ""evidence_span"": ""try some panadol"" } ], ""allergies"": [] }, ""objective"": { ""exam"": { ""text"": null, ""evidence_span"": null } }, ""assessment"": { ""problems"": [ { ""name"": ""tension headache"", ""evidence_span"": ""sounds like tension headache"" } ] }, ""plan"": { ""items"": [ { ""action"": ""rest"", ""evidence_span"": ""rest"" } ] } }","{ ""subjective"": { ""chief_complaint"": { ""text"": ""headache"", ""evidence_span"": ""headache"" }, ""hpi"": { ""text"": ""woke up with a terrible headache today"", ""evidence_span"": ""woke up with a terrible headache today"" }, ""pmh"": [], ""meds"": [ { ""text"": ""panadol"", ""evidence_span"": ""try some panadol"" } ], ""allergies"": [] }, ""objective"": { ""exam"": { ""text"": null, ""evidence_span"": null } }, ""assessment"": { ""problems"": [ { ""name"": ""tension headache"", ""evidence_span"": ""sounds like tension headache"" } ] }, ""plan"": { ""items"": [ { ""action"": ""rest"", ""evidence_span"": ""rest"" } ] } }"
transcript (truncated),gold,v2.1 prediction,v3 (DPO) prediction
"HCP: hello, i'm dr lee, how are you doing today Patient: hi dr lee, yeah i'm okay i suppose, just really tired all the time HCP: sounds like you've had a rough few weeks, what brings you in today Patient: well mostly the anxiety, i feel like i can't breathe when i'm in public HCP: that sounds really frightening, how long has this been going on Patient: probably about three months now, since my mum got sick, i just get panicky HCP: and has this affected your daily life at work or home Patient: yes, i can't leave the house much, i'm avoiding the shops and the bus stop HCP: i understand that's hard, have you noticed any changes in your sleep or appetite Patient: not really, i eat okay but i'm definitely not sleeping properly, waking up at 3am HCP: ok, and do you feel this is related to your mother's condition Patient: yeah i think so, i'm just scared something bad will happen to her HCP: do...","{ ""appearance"": { ""text"": null, ""evidence_span"": null }, ""behaviour"": { ""text"": ""social avoidance, agitated anxiety"", ""evidence_span"": ""i can't leave the house much, i'm avoiding the shops and the bus stop"" }, ""speech"": { ""text"": null, ""evidence_span"": null }, ""mood"": { ""text"": ""fearful, anxious"", ""evidence_span"": ""i feel like i can't breathe when i'm in public"" }, ""affect"": { ""text"": null, ""evidence_span

### Observation: Problem 2 after DPO

The v2.1 vs v3 table above, per in-distribution template:

| template | wrong_null_rate | ungrounded_span_rate | content_overlap | schema_valid |
|---|---|---|---|---|
| soap | 0.14 to 0.26 (worse, one collapse) | 0.01 to 0.00 (flat) | 0.52 to 0.46 (worse) | 1.00 to 0.90 (worse) |
| referral_a | 0.07 to 0.06 (better) | 0.05 to 0.03 (better) | 0.70 to 0.71 (flat) | 1.00 to 1.00 (held) |
| mse | 0.05 to 0.09 (worse) | 0.06 to 0.12 (worse) | 0.14 to 0.15 (flat) | 1.00 to 1.00 (held) |

referral_a is the one clean win: fewer misses and fewer hallucinations together, no schema cost. That is the Problem 2 result DPO was meant to deliver, and it shows the method is sound when the preference signal is strong and the training does not over-optimize. soap regressed through a single collapsed output on its longest transcript, and mse moved the wrong way on both misses and grounding.

The side-by-side diffs use the three v2.1 failure transcripts (soap index 2, mse index 5, mse index 2). On these v3 is essentially unchanged: soap index 2 still leaves 3 of 7 fields null, mse index 5 still emits 2 of 5 ungrounded spans. The measurable gain is the referral_a aggregate, which is not among the three picked transcripts, so the diffs understate it.

**Conclusion.** A partial result. DPO moved Problem 2 in the right direction on referral_a but not on soap or mse, and it is not yet net positive across the trained templates. Reported as-is. The recap and what-comes-next sections set out how to make the gain consistent.

## What SFT, DPO, and constrained decoding each do

It is easy to confuse these three, so here is what each one operates on and what it can and cannot fix. Worked on one real failure from above: the SOAP transcript (soap index 2) where v2.1 emitted valid JSON but left `hpi` null even though the patient described timing and severity.

| Technique | When it acts | What it operates on | Effect on the soap index 2 failure |
|---|---|---|---|
| **SFT** | training | per-token cross-entropy against one gold output | Taught the model the SOAP schema and general SOAP content patterns. Could not learn to prefer "populate when evidence is present", because that is a property of the whole output, not of any single token. This is why the `null` survives. |
| **Constrained decoding** (xgrammar / llama.cpp GBNF) | inference | a grammar mask over the next-token distribution | At each step it sets the logit of any token that would break the schema to negative infinity. It would guarantee parseable JSON, but `null` is a schema-valid value for `hpi`, so the mask still permits it. The three nulls would remain. It learns nothing. |
| **DPO** | training | preference between a better and a worse whole output | Given a pair (populated correctly = chosen, three nulls = rejected) on this transcript, DPO raises the probability of the chosen sequence relative to the rejected one. At inference the model then prefers opening the `hpi` object over emitting `null`. This is the gradient SFT structurally cannot produce. |

**The one-line distinction.**

- SFT learns from `(input, one target output)`.
- DPO learns from `(input, better output, worse output)`.
- Constrained decoding learns nothing. It is a runtime parse filter.

**On Problem 1 specifically.** Constrained decoding is sometimes proposed as the fix for unseen templates. It is not. On our canonical Problem 1 evidence, referral_b, `schema_valid` is already 1.00 across v1, v2, and v2.1 without any grammar. A grammar mask changes nothing there, because the gap is content (overlap stuck at 0.48), not JSON shape. Constrained decoding is orthogonal scaffolding that hardens the output contract for downstream consumers. It does not substitute for SFT coverage or DPO behaviour shaping.

## Recap

| Stage | What it shows | Problem class | Fix |
|---|---|---|---|
| v1 to v2 to v2.1, the basic capability table | Schema validity is solved by prompt-time injection. Content grows with each added training template. | n/a | SFT, more templates |
| referral_b stays flat at 0.48 even after MSE is added. MSE-on-v2 degenerates to nulls. | The model cannot handle templates it has never seen. | Problem 1, coverage | More training data on the missing template |
| v2.1 on trained templates has content misses and ungrounded spans inside schema-valid JSON. | Token loss cannot distinguish well-formed-but-wrong from well-formed-and-right. | Problem 2, behavioural | DPO on verifier-scored preference pairs |
| v3 (DPO, 2 epochs) improved referral_a (wrong_null 0.07 to 0.06, ungrounded 0.05 to 0.03) but regressed soap via one collapsed output (schema 1.00 to 0.90) and worsened mse. | Preference learning moves Problem 2 when the signal is strong, but the offline signal is weak and uneven and the longest SOAP target still collapses. | Problem 2, partly moved | Regularised on-policy DPO, then GRPO |

Problem 1 and Problem 2 are independent. DPO does not fix Problem 1. More SFT data does not fix Problem 2 cleanly. Offline DPO at 2 epochs moved Problem 2 on one template and broke another, a partial result that points straight at the fixes below.

## What comes next

**Phase 2, DPO, done and measured, a partial result.** v3 here is v2.1 further trained with TRL DPO on 210 verifier-scored preference pairs for 2 epochs, then exported to Q4_K_M GGUF and evaluated. It improved referral_a (wrong_null_rate 0.072 to 0.058 and ungrounded_span_rate 0.046 to 0.031, schema held), which is the exact Problem 2 movement the method targets. It also regressed soap through a single collapsed output on the longest transcript (schema_valid 1.00 to 0.90) and drifted the wrong way on mse. A longer 4-epoch run on a similar pair set was worse, collapsing three of ten SOAP outputs and erasing the referral_a gain, so training length is clearly driving the over-optimization. The signal is real but weak and uneven.

**Phase 2b, make the gain consistent.** In order:
1. Anchor the chosen likelihood. Add an NLL or SFT auxiliary term on the chosen response (RPO style, TRL `rpo_alpha`) so updates raise the chosen probability rather than only widening the margin by suppressing the rejected. This targets the single SOAP collapse directly.
2. Select on the verifier, not the loss. Checkpoint on the eval metrics (wrong_null_rate, ungrounded_span_rate, content_overlap with schema_valid held at 1.00). The 2-epoch model already beats the 4-epoch one, so early stopping on the verifier would have caught that.
3. Handle length. Ensure the training max length covers the longest SOAP targets so the richest transcripts are not truncated, since those are exactly the ones that collapse.
4. Strengthen the data. Sample chosen responses on-policy from v2.1 instead of using gold so the target is reachable, scale pairs with fresh synthetic transcripts rather than the eval set, and rebalance the rejected buckets so mse and soap are not dominated by null-miss rejections that share tokens with valid output.

**Phase 3, GRPO, the principled successor.** The remaining failures are properties of offline DPO, not of preference learning in general. DPO optimises a binary preference between two fixed, off-policy sequences and has no leash on absolute likelihood, which is what produced the SOAP collapse. GRPO removes those exact weaknesses. For each transcript it samples a group of completions from the current policy, scores each with the same verifier reward we already built (the scalar score, not a binary label), computes group-relative advantages, and optimises with a KL penalty to the reference. That makes the signal on-policy (no stale pairs, so no likelihood displacement), optimises the verifier score directly (so content_overlap and grounding get a gradient, not just a preference direction), and the KL anchor prevents the all-null collapse. The cost is compute, because GRPO generates inside the training loop and is heavier than offline DPO. The staged plan is therefore regularised on-policy DPO as the cheap fix, and GRPO when the offline signal proves too weak or too unstable.

**Non-goals, unchanged.** Neither DPO nor GRPO closes the referral_b ontology gap. Coverage problems need coverage solutions: more training data on the missing template, retrieval-style schema injection, or schema-conditioned synthetic data.